In [77]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

def calculate_pie_for_game(game_id):
    """Calculate PIE correctly by summing individual player contributions."""
    with engine.connect() as conn:
        query = text("""
            SELECT 
                player_id,
                pts, fgm, ftm, fga, fta, dreb, oreb,
                ast, stl, blk, pf, tov
            FROM player_game_stats
            WHERE game_id = :game_id
        """)
        
        players = conn.execute(query, {"game_id": game_id}).fetchall()
        
        game_total = 0
        player_contributions = {}
        
        for player in players:
            contribution = (
                (player.pts or 0) + 
                (player.fgm or 0) + 
                (player.ftm or 0) - 
                (player.fga or 0) - 
                (player.fta or 0) +
                (player.dreb or 0) + 
                (0.5 * (player.oreb or 0)) + 
                (player.ast or 0) + 
                (player.stl or 0) +
                (0.5 * (player.blk or 0)) - 
                (player.pf or 0) - 
                (player.tov or 0)
            )
            
            game_total += contribution
            player_contributions[player.player_id] = contribution
        
        pies = {}
        for player_id, contribution in player_contributions.items():
            pies[player_id] = (contribution / game_total * 100) if game_total != 0 else 0
            
        return pies


def calculate_adv_stats(game_id, player_id):
    """Calculate advanced stats for a player in a game."""
    
    with engine.connect() as conn:
        query = text("""
            SELECT 
                p.*,
                pt.team_fgm, pt.team_fga, pt.team_fg3m,
                pt.team_fta, pt.team_oreb, pt.team_dreb, 
                pt.team_reb, pt.team_tov, pt.team_min,
                ot.team_oreb as opp_oreb,
                ot.team_dreb as opp_dreb,
                ot.team_reb as opp_reb
            FROM player_game_stats p
            INNER JOIN team_game_stats pt 
                ON p.game_id = pt.game_id AND p.team_id = pt.team_id
            INNER JOIN team_game_stats ot 
                ON p.game_id = ot.game_id AND p.team_id != ot.team_id
            WHERE p.game_id = :game_id AND p.player_id = :player_id
        """)
        
        result = conn.execute(query, {"game_id": game_id, "player_id": player_id})
        row = result.fetchone()
        
        if not row:
            return None
        
        data = dict(row._mapping)
    
    # Extract stats
    player_min = data['min'] or 0
    player_pts = data['pts'] or 0
    player_fgm = data['fgm'] or 0
    player_fga = data['fga'] or 0
    player_fg3m = data['fg3m'] or 0
    player_ftm = data['ftm'] or 0
    player_fta = data['fta'] or 0
    player_oreb = data['oreb'] or 0
    player_dreb = data['dreb'] or 0
    player_reb = data['reb'] or 0
    player_ast = data['ast'] or 0
    player_tov = data['tov'] or 0
    
    team_min = data['team_min'] or 240
    team_fgm = data['team_fgm'] or 0
    team_fga = data['team_fga'] or 0
    team_fta = data['team_fta'] or 0
    team_oreb = data['team_oreb'] or 0
    team_dreb = data['team_dreb'] or 0
    team_reb = data['team_reb'] or 0
    team_tov = data['team_tov'] or 0
    
    opp_oreb = data['opp_oreb'] or 0
    opp_dreb = data['opp_dreb'] or 0
    opp_reb = data['opp_reb'] or 0
    
    adv_stats = {}
    
    # Shooting Efficiency
    if player_fga > 0:
        adv_stats['efg_pct'] = (player_fgm + 0.5 * player_fg3m) / player_fga
    else:
        adv_stats['efg_pct'] = None
    
    if (player_fga + player_fta) > 0:
        adv_stats['ts_pct'] = player_pts / (2 * (player_fga + 0.44 * player_fta))
    else:
        adv_stats['ts_pct'] = None
    
    # Playmaking
    if player_tov > 0:
        adv_stats['ast_tov'] = player_ast / player_tov
    else:
        adv_stats['ast_tov'] = None
    
    if player_min > 0 and team_min > 0:
        teammate_fgm = ((player_min / (team_min / 5)) * team_fgm) - player_fgm
        if teammate_fgm > 0:
            adv_stats['ast_pct'] = 100 * player_ast / teammate_fgm
        else:
            adv_stats['ast_pct'] = None
    else:
        adv_stats['ast_pct'] = None
    
    # Usage
    if player_min > 0 and team_min > 0:
        denominator = team_fga + 0.44 * team_fta + team_tov
        if denominator > 0:
            adv_stats['usg_pct'] = 100 * (
                (player_fga + 0.44 * player_fta + player_tov) * (team_min / 5)
            ) / (player_min * denominator)
        else:
            adv_stats['usg_pct'] = None
    else:
        adv_stats['usg_pct'] = None
    
    # Rebounding
    if player_min > 0 and team_min > 0:
        available_oreb = team_oreb + opp_dreb
        if available_oreb > 0:
            adv_stats['oreb_pct'] = 100 * (player_oreb * (team_min / 5)) / (
                player_min * available_oreb
            )
        else:
            adv_stats['oreb_pct'] = None
        
        available_dreb = team_dreb + opp_oreb
        if available_dreb > 0:
            adv_stats['dreb_pct'] = 100 * (player_dreb * (team_min / 5)) / (
                player_min * available_dreb
            )
        else:
            adv_stats['dreb_pct'] = None
        
        available_reb = team_reb + opp_reb
        if available_reb > 0:
            adv_stats['reb_pct'] = 100 * (player_reb * (team_min / 5)) / (
                player_min * available_reb
            )
        else:
            adv_stats['reb_pct'] = None
    else:
        adv_stats['oreb_pct'] = None
        adv_stats['dreb_pct'] = None
        adv_stats['reb_pct'] = None
    
    # PIE (CORRECTED - This is the key fix!)
    all_pies = calculate_pie_for_game(game_id)
    adv_stats['pie'] = all_pies.get(player_id, 0)
    
    return adv_stats

In [79]:
def debug_pie_calculation(game_id, player_id):
    """
    Debug helper to check if PIE is being calculated correctly.
    """
    from sqlalchemy import create_engine, text
    from dotenv import load_dotenv
    import os
    
    load_dotenv()
    DATABASE_URL = os.getenv("DATABASE_URL")
    engine = create_engine(DATABASE_URL)
    
    print(f"\n{'='*80}")
    print(f"PIE DEBUGGING FOR GAME {game_id}, PLAYER {player_id}")
    print(f"{'='*80}\n")
    
    with engine.connect() as conn:
        # Get the specific player's stats
        query = text("""
            SELECT pts, fgm, ftm, fga, fta, dreb, oreb, ast, stl, blk, pf, tov
            FROM player_game_stats
            WHERE game_id = :game_id AND player_id = :player_id
        """)
        player_row = conn.execute(query, {"game_id": game_id, "player_id": player_id}).fetchone()
        
        if not player_row:
            print("❌ Player not found!")
            return
        
        player_contribution = (
            (player_row.pts or 0) + 
            (player_row.fgm or 0) + 
            (player_row.ftm or 0) - 
            (player_row.fga or 0) - 
            (player_row.fta or 0) +
            (player_row.dreb or 0) + 
            (0.5 * (player_row.oreb or 0)) + 
            (player_row.ast or 0) + 
            (player_row.stl or 0) +
            (0.5 * (player_row.blk or 0)) - 
            (player_row.pf or 0) - 
            (player_row.tov or 0)
        )
        
        print(f"Player's contribution: {player_contribution:.2f}")
        print(f"  Components:")
        print(f"    PTS={player_row.pts}, FGM={player_row.fgm}, FTM={player_row.ftm}")
        print(f"    FGA={player_row.fga}, FTA={player_row.fta}")
        print(f"    DREB={player_row.dreb}, OREB={player_row.oreb}")
        print(f"    AST={player_row.ast}, STL={player_row.stl}, BLK={player_row.blk}")
        print(f"    PF={player_row.pf}, TOV={player_row.tov}")
        
        # Get all players to calculate game total
        query = text("""
            SELECT COUNT(*) as player_count,
                   SUM(pts) as total_pts,
                   SUM(fgm) as total_fgm,
                   SUM(pts + fgm + ftm - fga - fta + dreb + (0.5 * oreb) + ast + stl + (0.5 * blk) - pf - tov) as game_total
            FROM player_game_stats
            WHERE game_id = :game_id
        """)
        game_row = conn.execute(query, {"game_id": game_id}).fetchone()
        
        print(f"\nGame totals:")
        print(f"  Players in game: {game_row.player_count}")
        print(f"  Total points: {game_row.total_pts}")
        print(f"  Game contribution total: {game_row.game_total:.2f}")
        
        if game_row.game_total > 0:
            pie = (player_contribution / game_row.game_total) * 100
            print(f"\nCalculated PIE: {pie:.3f}")
            print(f"Expected NBA PIE: ~18-19 for this example")
            
            if abs(pie - 18.9) < 1:
                print("✅ PIE calculation looks CORRECT!")
            else:
                print("❌ PIE calculation still appears WRONG")
                print("   Check that you're using calculate_pie_for_game() function")
        
    print(f"{'='*80}\n")

def compare_with_nba_stats(our_stats, nba_json, verbose=False):
    """
    Compare our calculated stats with NBA's official stats from JSON.
    
    Args:
        our_stats: Dictionary of our calculated advanced stats (PLAYER stats only)
        nba_json: List/dict from NBA API (or single dict)
        verbose: If True, print detailed debugging info
    """
    import json
    
    # Handle if it's a list, take first element
    if isinstance(nba_json, list):
        nba_data = nba_json[0]
    else:
        nba_data = nba_json
    
    # Mapping of NBA stat names to our stat names
    # NOTE: OFF_RATING, DEF_RATING, NET_RATING, PACE are TEAM stats
    # NBA reports INDIVIDUAL PLAYER ratings which we cannot calculate from box scores
    stat_mapping = {
        'EFG_PCT': 'efg_pct',
        'TS_PCT': 'ts_pct',
        'USG_PCT': 'usg_pct',
        'AST_PCT': 'ast_pct',
        'OREB_PCT': 'oreb_pct',
        'DREB_PCT': 'dreb_pct',
        'REB_PCT': 'reb_pct',
        'AST_TOV': 'ast_tov',
        'PIE': 'pie'
        # Removed: OFF_RATING, DEF_RATING, NET_RATING, PACE (these are team stats)
    }
    
    print(f"\n{'='*80}")
    print(f"STAT COMPARISON - Game: {nba_data.get('GAME_ID')}, Player: {nba_data.get('PLAYER_NAME')}")
    print(f"{'='*80}\n")
    
    if verbose:
        print("DEBUG INFO:")
        print(f"  Player ID: {nba_data.get('PLAYER_ID')}")
        print(f"  Minutes: {nba_data.get('MIN')}")
        print(f"  Our stats keys: {list(our_stats.keys())}")
        print()
    
    print(f"{'STAT':<20} {'NBA VALUE':<20} {'OUR VALUE':<20} {'DIFFERENCE':<20}")
    print(f"{'-'*80}")
    
    comparisons = []
    
    for nba_key, our_key in stat_mapping.items():
        if nba_key in nba_data and our_key in our_stats:
            nba_val = nba_data[nba_key]
            our_val = our_stats[our_key]
            
            # Skip if either is None
            if nba_val is None or our_val is None:
                continue
            
            # Convert NBA percentages from decimal (0-1) to percentage (0-100) if needed
            # NBA stores some as decimals, some as percentages
            if nba_key in ['EFG_PCT', 'TS_PCT']:
                # These are decimals (0-1)
                nba_val_display = nba_val * 100
                our_val_display = our_val * 100 if our_val < 2 else our_val
            elif nba_key in ['AST_PCT', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'USG_PCT']:
                # These are already percentages (0-100) but might be decimals
                nba_val_display = nba_val if nba_val >= 1 else nba_val * 100
                our_val_display = our_val if our_val >= 1 else our_val * 100
            elif nba_key == 'PIE':
                # PIE can be decimal (0-1) or percentage (0-100) depending on endpoint
                nba_val_display = nba_val if nba_val >= 1 else nba_val * 100
                our_val_display = our_val if our_val >= 1 else our_val * 100
            else:
                nba_val_display = nba_val
                our_val_display = our_val
            
            if verbose:
                print(f"DEBUG {nba_key}: NBA={nba_val} -> {nba_val_display}, Ours={our_val} -> {our_val_display}")
            
            # Calculate difference
            diff = our_val_display - nba_val_display
            diff_pct = (diff / nba_val_display * 100) if nba_val_display != 0 else 0
            
            # Format based on stat type
            if nba_key in ['EFG_PCT', 'TS_PCT']:
                nba_str = f"{nba_val_display:.3f}%"
                our_str = f"{our_val_display:.3f}%"
                diff_str = f"{diff:+.3f}% ({diff_pct:+.1f}%)"
            elif nba_key in ['AST_PCT', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'USG_PCT']:
                nba_str = f"{nba_val_display:.1f}%"
                our_str = f"{our_val_display:.1f}%"
                diff_str = f"{diff:+.1f}% ({diff_pct:+.1f}%)"
            elif nba_key == 'PIE':
                nba_str = f"{nba_val_display:.3f}"
                our_str = f"{our_val_display:.3f}"
                diff_str = f"{diff:+.3f} ({diff_pct:+.1f}%)"
            elif nba_key == 'AST_TOV':
                nba_str = f"{nba_val_display:.2f}"
                our_str = f"{our_val_display:.2f}"
                diff_str = f"{diff:+.2f} ({diff_pct:+.1f}%)"
            else:
                nba_str = f"{nba_val_display:.1f}"
                our_str = f"{our_val_display:.1f}"
                diff_str = f"{diff:+.1f} ({diff_pct:+.1f}%)"
            
            print(f"{nba_key:<20} {nba_str:<20} {our_str:<20} {diff_str:<20}")
            
            comparisons.append({
                'stat': nba_key,
                'nba': nba_val_display,
                'ours': our_val_display,
                'diff': diff,
                'diff_pct': diff_pct
            })
    
    # Note about team stats
    print(f"\n{'-'*80}")
    print(f"\n⚠️  TEAM-LEVEL STATS (not calculated for individual players):")
    print(f"{'-'*80}")
    team_stats = ['OFF_RATING', 'DEF_RATING', 'NET_RATING', 'PACE']
    for stat in team_stats:
        if stat in nba_data:
            val = nba_data[stat]
            if val is not None:
                print(f"  - {stat:<20} = {val:<10.1f} (requires play-by-play data)")
    
    # Find other stats NBA has that we don't calculate
    print(f"\n{'-'*80}")
    print(f"\nOTHER STATS IN NBA DATA:")
    print(f"{'-'*80}")
    
    ignore_keys = [
        'index', 'GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_CITY',
        'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'START_POSITION', 'COMMENT', 'MIN',
        'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'PACE'  # Team stats
    ]
    
    nba_only_stats = []
    for key in nba_data.keys():
        if key not in stat_mapping and key not in ignore_keys:
            nba_only_stats.append(key)
            val = nba_data[key]
            if val is not None:
                print(f"  - {key:<25} = {val}")
    
    # Summary
    print(f"\n{'='*80}")
    print(f"SUMMARY:")
    print(f"  Stats compared:              {len(comparisons)}")
    print(f"  Other NBA stats:             {len(nba_only_stats)}")
    
    # Check for large discrepancies
    large_diffs = [c for c in comparisons if abs(c['diff_pct']) > 5]
    if large_diffs:
        print(f"  ⚠️  Large discrepancies (>5%): {len(large_diffs)}")
        for c in large_diffs:
            print(f"      - {c['stat']}: {c['diff_pct']:+.1f}% off")
    else:
        print(f"  ✓ All stats within 5% tolerance")
    
    # Special check for PIE
    pie_comparison = next((c for c in comparisons if c['stat'] == 'PIE'), None)
    if pie_comparison and abs(pie_comparison['diff_pct']) > 10:
        print(f"\n  ❌ PIE ERROR: {pie_comparison['diff_pct']:.1f}% off!")
        print(f"     This likely means you're still using the OLD formula.")
        print(f"     Make sure you're calling calculate_adv_stats() with the CORRECTED function.")
    
    print(f"{'='*80}\n")
    
    return comparisons, nba_only_stats

In [85]:
# Add to your main section:
if __name__ == "__main__":
    # Test with the sample game/player
    stats = calculate_adv_stats(game_id='0022301147', player_id=1630552)
    
    if stats:
        # Your existing pretty print...
        print(f"\n{'='*60}")
        print(f"ADVANCED STATS - Game: 0022301147, Player: 1630552")
        print(f"{'='*60}\n")
        
        print("SHOOTING EFFICIENCY:")
        print(f"  EFG%:          {stats['efg_pct']:.3f}" if stats['efg_pct'] else "  EFG%:          N/A")
        print(f"  TS%:           {stats['ts_pct']:.3f}" if stats['ts_pct'] else "  TS%:           N/A")
        
        print("\nUSAGE & PERCENTAGES:")
        print(f"  USG%:          {stats['usg_pct']:.1f}%" if stats['usg_pct'] else "  USG%:          N/A")
        print(f"  AST%:          {stats['ast_pct']:.1f}%" if stats['ast_pct'] else "  AST%:          N/A")
        print(f"  OREB%:         {stats['oreb_pct']:.1f}%" if stats['oreb_pct'] else "  OREB%:         N/A")
        print(f"  DREB%:         {stats['dreb_pct']:.1f}%" if stats['dreb_pct'] else "  DREB%:         N/A")
        print(f"  REB%:          {stats['reb_pct']:.1f}%" if stats['reb_pct'] else "  REB%:          N/A")
        print(f"  AST/TOV:       {stats['ast_tov']:.2f}" if stats['ast_tov'] else "  AST/TOV:       N/A")
        
        print("\nPACE & RATINGS:")
        # print(f"  PACE:          {stats['pace']:.1f}" if stats['pace'] else "  PACE:          N/A")
        # print(f"  OFF Rating:    {stats['off_rating']:.1f}" if stats['off_rating'] else "  OFF Rating:    N/A")
        # print(f"  DEF Rating:    {stats['def_rating']:.1f}" if stats['def_rating'] else "  DEF Rating:    N/A")
        # print(f"  NET Rating:    {stats['net_rating']:.1f}" if stats['net_rating'] else "  NET Rating:    N/A")
        
        print("\nIMPACT:")
        print(f"  PIE:           {stats['pie']:.3f}" if stats['pie'] else "  PIE:           N/A")
        
        print(f"\n{'='*60}\n")
        
        # Now compare with NBA's official stats
        nba_json = [
          {
            "index": 1474,
            "GAME_ID": "0022301147",
            "TEAM_ID": 1610612737,
            "TEAM_ABBREVIATION": "ATL",
            "TEAM_CITY": "Atlanta",
            "PLAYER_ID": 1630552,
            "PLAYER_NAME": "Jalen Johnson",
            "NICKNAME": "Jalen",
            "START_POSITION": "F",
            "COMMENT": "",
            "MIN": "23.000000:43",
            "E_OFF_RATING": 88,
            "OFF_RATING": 91.7,
            "E_DEF_RATING": 122.9,
            "DEF_RATING": 123.4,
            "E_NET_RATING": -34.9,
            "NET_RATING": -31.7,
            "AST_PCT": 0.214,
            "AST_TOV": 3,
            "AST_RATIO": 27.3,
            "OREB_PCT": 0.063,
            "DREB_PCT": 0.353,
            "REB_PCT": 0.163,
            "TM_TOV_PCT": 9.1,
            "EFG_PCT": 0.929,
            "TS_PCT": 0.929,
            "USG_PCT": 0.131,
            "E_USG_PCT": 0.14,
            "E_PACE": 98.36,
            "PACE": 96.13,
            "PACE_PER40": 80.11,
            "POSS": 48,
            "PIE": 0.189
          }
        ]
        
        compare_with_nba_stats(stats, nba_json)


ADVANCED STATS - Game: 0022301147, Player: 1630552

SHOOTING EFFICIENCY:
  EFG%:          0.929
  TS%:           0.929

USAGE & PERCENTAGES:
  USG%:          14.5%
  AST%:          24.0%
  OREB%:         8.4%
  DREB%:         31.1%
  REB%:          18.5%
  AST/TOV:       3.00

PACE & RATINGS:

IMPACT:
  PIE:           9.639



STAT COMPARISON - Game: 0022301147, Player: Jalen Johnson

STAT                 NBA VALUE            OUR VALUE            DIFFERENCE          
--------------------------------------------------------------------------------
EFG_PCT              92.900%              92.857%              -0.043% (-0.0%)     
TS_PCT               92.900%              92.857%              -0.043% (-0.0%)     
USG_PCT              13.1%                14.5%                +1.4% (+10.8%)      
AST_PCT              21.4%                24.0%                +2.6% (+12.2%)      
OREB_PCT             6.3%                 8.4%                 +2.1% (+33.2%)      
DREB_PCT             35.3%